# Schema Design: Vaccination Booking System

The trust needs a new vaccination booking system. No legacy data, no spreadsheets -- a blank slate. Your job: design the schema.

This is a different challenge from normalising existing data or building an analytical warehouse. Here we are starting from requirements, thinking about entities and relationships, and making deliberate choices about keys, constraints, and data integrity.

Every constraint you add is a rule the database enforces automatically. Every constraint you leave out is a rule that relies on application code, human discipline, or luck. In healthcare, luck is not a strategy.

## Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)

In [ ]:
%pip install -q jupysql
%load_ext sql

In [ ]:
%sql sqlite:///../data/nhs_trust.sqlite

## Requirements gathering

Before writing any SQL, we need to understand what the system must track. Here is what the vaccination programme manager tells us:

- **Patients** book appointments for specific **vaccines** (COVID booster, flu, shingles, etc.)
- Appointments happen at specific **locations** (community centres, pharmacies, hospital clinics)
- Each appointment is administered by a **staff member** (nurse, pharmacist, doctor)
- We need to track which **batch** of vaccine was used (for recall purposes)
- Patients might have **multiple appointments** for different vaccines
- Some vaccines require **multiple doses** -- we need to track dose number
- An appointment can be booked, completed, cancelled, or no-show

From this, we can identify our entities:

1. **Patients** -- the people being vaccinated
2. **Vaccines** -- the types of vaccine available
3. **Locations** -- where vaccinations happen
4. **Staff** -- who administers the vaccines
5. **Vaccine batches** -- specific batches for traceability
6. **Appointments** -- the central entity linking everything together

## Entity-relationship thinking

Before we write CREATE TABLE statements, let's think about how these entities relate.

```
patients ----< appointments >---- staff
                   |    |
                   v    v
              locations  vaccine_batches >---- vaccines
```

Reading the relationships:

- One **patient** can have many **appointments** (one-to-many)
- One **staff member** can administer many **appointments** (one-to-many)
- Each **appointment** happens at one **location** (many-to-one)
- Each **appointment** uses one **vaccine batch** (many-to-one)
- Each **vaccine batch** is of one **vaccine** type (many-to-one)

The appointment table is the centre of gravity. It is the *event* table -- the thing that actually happens. Everything else is reference data that describes the event.

## Natural vs surrogate keys

A quick but important design decision.

A **natural key** is a real-world identifier: NHS number, vaccine batch number, postcode. It has meaning outside the database.

A **surrogate key** is a database-generated identifier: an auto-incrementing integer or a UUID. It has no real-world meaning.

| | Natural key | Surrogate key |
|---|---|---|
| **Example** | NHS number (`4857293016`) | `patient_id = 42` |
| **Meaningful** | Yes -- you can look it up | No -- just a row identifier |
| **Stable** | Maybe not -- NHS numbers can have errors | Yes -- never changes |
| **Size** | Variable (strings, composites) | Fixed (integer) |
| **JOINs** | Slower (string comparison) | Faster (integer comparison) |

The pragmatic answer: **use surrogate keys as primary keys, but store natural keys as UNIQUE constraints**.

This gives you the best of both worlds. The surrogate key handles foreign key relationships efficiently. The UNIQUE constraint on the natural key ensures you cannot accidentally create two records for the same real-world entity.

For patients, this means:
- `patient_id INTEGER PRIMARY KEY` -- the surrogate key, used in JOINs
- `nhs_number TEXT NOT NULL UNIQUE` -- the natural key, enforcing uniqueness

## Building the schema

Let's create each table with appropriate constraints. We will use SQLite, but the principles apply to any relational database.

First, we need to enable foreign key enforcement in SQLite (it is off by default -- a quirk worth knowing):

In [ ]:
%sql PRAGMA foreign_keys = ON;

### The vaccines table

This is reference data -- the types of vaccine available.

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS vaccines (
    vaccine_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL UNIQUE,
    manufacturer TEXT NOT NULL,
    doses_required INTEGER NOT NULL CHECK (doses_required >= 1),
    min_dose_interval_days INTEGER  -- NULL for single-dose vaccines
);

Notice the constraints:

- `NOT NULL` on name, manufacturer, doses_required -- these must always have a value
- `UNIQUE` on name -- you cannot have two vaccines with the same name
- `CHECK (doses_required >= 1)` -- a vaccine must require at least one dose
- `min_dose_interval_days` is nullable -- single-dose vaccines do not need it

In [ ]:
%%sql
INSERT OR IGNORE INTO vaccines (name, manufacturer, doses_required, min_dose_interval_days) VALUES
    ('COVID-19 Autumn Booster', 'Pfizer-BioNTech', 1, NULL),
    ('Flu (2024/25)', 'Sanofi Pasteur', 1, NULL),
    ('Shingles (Shingrix)', 'GSK', 2, 60),
    ('Pneumococcal (PPV23)', 'Merck', 1, NULL),
    ('Tdap Booster', 'Sanofi Pasteur', 1, NULL),
    ('HPV (Gardasil 9)', 'Merck', 2, 180),
    ('Hepatitis B', 'GSK', 3, 28);

In [ ]:
%sql SELECT * FROM vaccines;

### The locations table

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS vax_locations (
    location_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    address TEXT NOT NULL,
    postcode TEXT NOT NULL,
    location_type TEXT NOT NULL CHECK (location_type IN ('Hospital', 'Community Centre', 'Pharmacy', 'GP Surgery')),
    capacity_per_hour INTEGER NOT NULL CHECK (capacity_per_hour > 0)
);

The `CHECK` constraint on `location_type` acts like an enum -- it restricts values to a known set. If someone tries to insert `location_type = 'Car Park'`, the database rejects it.

In [ ]:
%%sql
INSERT OR IGNORE INTO vax_locations (name, address, postcode, location_type, capacity_per_hour) VALUES
    ('Millbrook Community Centre', '15 High Street, Millbrook', 'CB1 3QR', 'Community Centre', 30),
    ('Ashford Pharmacy', '42 Station Road, Ashford', 'PE3 7TH', 'Pharmacy', 8),
    ('Trust Hospital Main Hall', '1 Hospital Way, Westbury', 'CB2 1AB', 'Hospital', 50),
    ('Riverside GP Surgery', '29 High Street, Lakeside', 'IP4 2NR', 'GP Surgery', 12),
    ('Northfield Library Hall', '88 Victoria Road, Northfield', 'NR5 9PL', 'Community Centre', 25);

In [ ]:
%sql SELECT * FROM vax_locations;

### The vaccination staff table

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS vax_staff (
    staff_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    role TEXT NOT NULL CHECK (role IN ('Nurse', 'Pharmacist', 'Doctor', 'Healthcare Assistant')),
    registration_number TEXT NOT NULL UNIQUE  -- NMC, GPhC, or GMC number
);

In [ ]:
%%sql
INSERT OR IGNORE INTO vax_staff (name, role, registration_number) VALUES
    ('Nurse Priya Patel', 'Nurse', 'NMC-20A1234'),
    ('Nurse James Cooper', 'Nurse', 'NMC-19B5678'),
    ('Dr Sarah Mitchell', 'Doctor', 'GMC-7654321'),
    ('Pharmacist Wei Chen', 'Pharmacist', 'GPhC-2098765'),
    ('Nurse Fatima Begum', 'Nurse', 'NMC-21C9012'),
    ('HCA David Thompson', 'Healthcare Assistant', 'HCA-2024-0042');

In [ ]:
%sql SELECT * FROM vax_staff;

### Vaccine batches

Traceability is critical. If a batch is recalled, we need to know exactly which patients received it.

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS vaccine_batches (
    batch_id INTEGER PRIMARY KEY AUTOINCREMENT,
    batch_number TEXT NOT NULL UNIQUE,
    vaccine_id INTEGER NOT NULL,
    expiry_date TEXT NOT NULL,
    doses_in_batch INTEGER NOT NULL CHECK (doses_in_batch > 0),
    FOREIGN KEY (vaccine_id) REFERENCES vaccines(vaccine_id)
);

In [ ]:
%%sql
INSERT OR IGNORE INTO vaccine_batches (batch_number, vaccine_id, expiry_date, doses_in_batch) VALUES
    ('PFZ-2024-A001', 1, '2025-03-31', 600),
    ('PFZ-2024-A002', 1, '2025-04-15', 600),
    ('SNF-FLU-24-B010', 2, '2025-06-30', 1000),
    ('GSK-SHN-24-C005', 3, '2025-09-15', 200),
    ('MRK-PPV-24-D003', 4, '2026-01-31', 500),
    ('SNF-TDAP-24-E001', 5, '2025-12-31', 400),
    ('MRK-HPV-24-F002', 6, '2025-08-31', 300),
    ('GSK-HEP-24-G001', 7, '2025-11-30', 250);

In [ ]:
%%sql
SELECT b.batch_number, v.name AS vaccine, b.expiry_date, b.doses_in_batch
FROM vaccine_batches b
JOIN vaccines v ON b.vaccine_id = v.vaccine_id;

### The appointments table

This is the main event table. It ties together patients, locations, staff, and vaccine batches.

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS vax_appointments (
    appointment_id INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id INTEGER NOT NULL,
    vaccine_id INTEGER NOT NULL,
    dose_number INTEGER NOT NULL CHECK (dose_number >= 1),
    location_id INTEGER NOT NULL,
    staff_id INTEGER,                   -- NULL if not yet assigned
    batch_id INTEGER,                   -- NULL until vaccine is administered
    appointment_date TEXT NOT NULL,
    appointment_time TEXT NOT NULL,
    status TEXT NOT NULL DEFAULT 'Booked'
        CHECK (status IN ('Booked', 'Completed', 'Cancelled', 'No-show')),
    notes TEXT,
    FOREIGN KEY (patient_id) REFERENCES patients(patient_id),
    FOREIGN KEY (vaccine_id) REFERENCES vaccines(vaccine_id),
    FOREIGN KEY (location_id) REFERENCES vax_locations(location_id),
    FOREIGN KEY (staff_id) REFERENCES vax_staff(staff_id),
    FOREIGN KEY (batch_id) REFERENCES vaccine_batches(batch_id)
);

Let's examine the design choices here:

- `patient_id` is NOT NULL -- you cannot book an appointment without a patient.
- `staff_id` is nullable -- the staff member might not be assigned until the day.
- `batch_id` is nullable -- we do not know which batch until the vaccine is actually administered.
- `status` has a DEFAULT and a CHECK constraint -- new rows are automatically 'Booked', and only valid statuses are accepted.
- `dose_number` has a CHECK -- no zero or negative doses.
- Five FOREIGN KEY constraints ensure every reference points to a real record.

Each of these constraints is a decision about data integrity. Together, they define the rules of the system.

## Inserting some appointments

Let's book some appointments to see the schema in action.

In [ ]:
%%sql
INSERT OR IGNORE INTO vax_appointments
    (patient_id, vaccine_id, dose_number, location_id, staff_id, batch_id,
     appointment_date, appointment_time, status, notes)
VALUES
    -- Completed COVID boosters
    (1, 1, 1, 3, 1, 1, '2024-10-01', '09:00', 'Completed', NULL),
    (2, 1, 1, 3, 1, 1, '2024-10-01', '09:15', 'Completed', NULL),
    (3, 1, 1, 1, 2, 2, '2024-10-05', '10:00', 'Completed', NULL),

    -- Flu vaccinations
    (1, 2, 1, 2, 4, 3, '2024-10-15', '14:00', 'Completed', NULL),
    (4, 2, 1, 2, 4, 3, '2024-10-15', '14:15', 'No-show', 'Patient did not attend'),

    -- Shingles - first and second doses
    (5, 3, 1, 3, 3, 4, '2024-09-01', '11:00', 'Completed', NULL),
    (5, 3, 2, 3, 3, 4, '2024-12-01', '11:00', 'Booked', NULL),

    -- Hepatitis B - three dose course
    (10, 7, 1, 4, 5, 8, '2024-08-01', '09:30', 'Completed', NULL),
    (10, 7, 2, 4, 5, 8, '2024-09-01', '09:30', 'Completed', NULL),
    (10, 7, 3, 4, 5, 8, '2024-10-01', '09:30', 'Booked', NULL),

    -- Cancelled appointment
    (15, 1, 1, 1, NULL, NULL, '2024-11-01', '10:00', 'Cancelled', 'Patient requested cancellation'),

    -- Future bookings (no staff or batch assigned yet)
    (20, 2, 1, 5, NULL, NULL, '2024-12-15', '14:00', 'Booked', NULL),
    (25, 4, 1, 3, NULL, NULL, '2024-12-20', '10:30', 'Booked', NULL);

In [ ]:
%%sql
SELECT
    a.appointment_id,
    p.name AS patient,
    v.name AS vaccine,
    a.dose_number,
    l.name AS location,
    a.appointment_date,
    a.status
FROM vax_appointments a
JOIN patients p ON a.patient_id = p.patient_id
JOIN vaccines v ON a.vaccine_id = v.vaccine_id
JOIN vax_locations l ON a.location_id = l.location_id
ORDER BY a.appointment_date, a.appointment_time;

## The foreign key as safety net

Here is the moment that justifies every constraint we wrote.

Suppose someone tries to book an appointment for a patient that does not exist -- perhaps a typo in the patient ID, perhaps a bug in the booking application.

In [ ]:
%%sql
-- This SHOULD fail: patient_id 9999 does not exist
INSERT INTO vax_appointments
    (patient_id, vaccine_id, dose_number, location_id, appointment_date, appointment_time, status)
VALUES
    (9999, 1, 1, 1, '2024-12-25', '09:00', 'Booked');

The database rejected it. The foreign key constraint on `patient_id` checked whether patient 9999 exists in the `patients` table, found nothing, and refused the INSERT.

Now let's see what happens if we **disable** foreign key enforcement and try the same thing:

In [ ]:
%%sql
PRAGMA foreign_keys = OFF;

In [ ]:
%%sql
-- With foreign keys OFF, this succeeds -- and creates an orphan
INSERT INTO vax_appointments
    (patient_id, vaccine_id, dose_number, location_id, appointment_date, appointment_time, status)
VALUES
    (9999, 1, 1, 1, '2024-12-25', '09:00', 'Booked');

In [ ]:
%%sql
-- The orphan appointment now exists
SELECT a.appointment_id, a.patient_id, a.appointment_date, a.status
FROM vax_appointments a
LEFT JOIN patients p ON a.patient_id = p.patient_id
WHERE p.patient_id IS NULL;

There it is. An appointment booked for a patient who does not exist. A vaccine dose allocated to nobody. A slot taken up that could have gone to a real person.

Without the foreign key constraint, this happened silently. No error, no warning. The booking application would have shown a success message. The nurse would show up on Christmas Day to vaccinate thin air.

Let's clean up the orphan and re-enable foreign keys:

In [ ]:
%%sql
DELETE FROM vax_appointments WHERE patient_id = 9999;
PRAGMA foreign_keys = ON;

## Constraint-driven design

Let's test a few more constraints.

### CHECK constraint: invalid status

In [ ]:
%%sql
-- This should fail: 'Pending' is not in the allowed status values
INSERT INTO vax_appointments
    (patient_id, vaccine_id, dose_number, location_id, appointment_date, appointment_time, status)
VALUES
    (1, 1, 1, 1, '2024-12-25', '09:00', 'Pending');

Rejected. The CHECK constraint on `status` only allows 'Booked', 'Completed', 'Cancelled', and 'No-show'.

### CHECK constraint: zero doses

In [ ]:
%%sql
-- This should fail: dose_number must be >= 1
INSERT INTO vax_appointments
    (patient_id, vaccine_id, dose_number, location_id, appointment_date, appointment_time, status)
VALUES
    (1, 1, 0, 1, '2024-12-25', '09:00', 'Booked');

### UNIQUE constraint: duplicate vaccine names

In [ ]:
%%sql
-- This should fail: vaccine name must be unique
INSERT INTO vaccines (name, manufacturer, doses_required)
VALUES ('Flu (2024/25)', 'Different Manufacturer', 1);

Every one of these errors was caught at the database level. The application code does not need to check -- the database enforces the rules. If a junior developer writes a buggy booking form, the data stays clean.

## Validation queries

Even with constraints, it is good practice to write validation queries that check data quality. These are useful for monitoring and auditing.

### Find orphan records

In [ ]:
%%sql
-- Check for appointments referencing non-existent patients
SELECT a.appointment_id, a.patient_id
FROM vax_appointments a
LEFT JOIN patients p ON a.patient_id = p.patient_id
WHERE p.patient_id IS NULL;

Good -- no orphans. Our foreign key constraint is doing its job.

### Check for expired batches being used

In [ ]:
%%sql
-- Find completed appointments that used an expired batch
SELECT
    a.appointment_id,
    a.appointment_date,
    b.batch_number,
    b.expiry_date,
    v.name AS vaccine
FROM vax_appointments a
JOIN vaccine_batches b ON a.batch_id = b.batch_id
JOIN vaccines v ON b.vaccine_id = v.vaccine_id
WHERE a.status = 'Completed'
  AND a.appointment_date > b.expiry_date;

### Check dose sequencing

In [ ]:
%%sql
-- Find patients booked for dose 2+ without a completed dose 1
SELECT
    a.patient_id,
    p.name,
    v.name AS vaccine,
    a.dose_number,
    a.status
FROM vax_appointments a
JOIN patients p ON a.patient_id = p.patient_id
JOIN vaccines v ON a.vaccine_id = v.vaccine_id
WHERE a.dose_number > 1
  AND NOT EXISTS (
      SELECT 1 FROM vax_appointments prev
      WHERE prev.patient_id = a.patient_id
        AND prev.vaccine_id = a.vaccine_id
        AND prev.dose_number = a.dose_number - 1
        AND prev.status = 'Completed'
  );

### Summary of available appointments by vaccine and location

In [ ]:
%%sql
SELECT
    v.name AS vaccine,
    l.name AS location,
    COUNT(*) AS total_appointments,
    SUM(CASE WHEN a.status = 'Completed' THEN 1 ELSE 0 END) AS completed,
    SUM(CASE WHEN a.status = 'Booked' THEN 1 ELSE 0 END) AS booked,
    SUM(CASE WHEN a.status = 'No-show' THEN 1 ELSE 0 END) AS no_shows,
    SUM(CASE WHEN a.status = 'Cancelled' THEN 1 ELSE 0 END) AS cancelled
FROM vax_appointments a
JOIN vaccines v ON a.vaccine_id = v.vaccine_id
JOIN vax_locations l ON a.location_id = l.location_id
GROUP BY v.name, l.name
ORDER BY v.name, l.name;

## Batch recall: a traceability query

One of the most important reasons to track batch numbers is recall. If batch `PFZ-2024-A001` is found to be defective, we need to know every patient who received it.

In [ ]:
%%sql
SELECT
    p.name AS patient_name,
    p.nhs_number,
    a.appointment_date,
    v.name AS vaccine,
    b.batch_number
FROM vax_appointments a
JOIN patients p ON a.patient_id = p.patient_id
JOIN vaccines v ON a.vaccine_id = v.vaccine_id
JOIN vaccine_batches b ON a.batch_id = b.batch_id
WHERE b.batch_number = 'PFZ-2024-A001'
  AND a.status = 'Completed'
ORDER BY a.appointment_date;

Within seconds, you have a list of every affected patient, their NHS number, and the date they were vaccinated. Without this schema design, you would be digging through paper records.

## Summary

| Design principle | Why it matters |
|---|---|
| **Entity-relationship thinking** | Identify entities and their relationships before writing SQL |
| **Surrogate keys + natural key UNIQUE** | Efficient JOINs plus real-world uniqueness enforcement |
| **NOT NULL** | Force essential data to be present |
| **UNIQUE** | Prevent duplicate real-world entities |
| **CHECK** | Restrict values to valid ranges or sets |
| **FOREIGN KEY** | Prevent orphan records -- the last line of defence |
| **Validation queries** | Defence in depth -- monitor data quality even with constraints |

The schema is the contract between your application and your data. Every constraint is a promise: this data will always be valid. The more promises the schema makes, the less your application code has to worry about, and the fewer ways things can go wrong.

In healthcare, where data errors can have real consequences for real people, that matters.

---

## Exercises

### Exercise 1: Add an adverse reactions table

After vaccination, some patients report adverse reactions (sore arm, headache, fever, anaphylaxis). Design and create an `adverse_reactions` table that:

- Links to the specific appointment
- Records the reaction type, severity (mild/moderate/severe), date reported, and any notes
- Uses appropriate constraints (NOT NULL, CHECK, FOREIGN KEY)

Then insert at least three sample records.

In [ ]:
%%sql

-- YOUR SQL HERE


### Exercise 2: Write a no-show analysis query

Write a query to find the **no-show rate by location**. Which locations have the highest proportion of no-shows? What might explain the differences?

In [ ]:
%%sql

-- YOUR SQL HERE


### Exercise 3: Dose interval validation

Write a validation query that finds patients whose second dose was booked **too soon** after their first dose, based on the `min_dose_interval_days` in the `vaccines` table.

Hint: you will need to JOIN `vax_appointments` to itself (a self-join) to compare dose 1 and dose 2 dates for the same patient and vaccine.

In [ ]:
%%sql

-- YOUR SQL HERE


### Exercise 4: Add a consent table

NHS vaccinations require informed consent. Design a `consent_records` table that:

- Links to the appointment
- Records whether consent was given (yes/no/withdrawn)
- Records who gave consent (patient, parent/guardian, lasting power of attorney)
- Records the date and time consent was recorded
- Ensures every completed appointment has a consent record

Think carefully about which constraints enforce the business rules. Write the CREATE TABLE, INSERT some data, then write a validation query to find completed appointments without consent records.

In [ ]:
%%sql

-- YOUR SQL HERE


### Exercise 5: Batch stock monitoring

Write a query that shows, for each vaccine batch:

- The batch number and vaccine name
- The total doses in the batch
- The number of doses used (completed appointments)
- The remaining doses
- Whether the batch has expired

Order by remaining doses ascending so that low-stock batches appear first.

In [ ]:
%%sql

-- YOUR SQL HERE
